# DentBid - train the damage and car-parts models

Runtime > Change runtime type > **GPU** (T4 or better).

Dataset: **CarDD** (4,000 images, 9,000+ labelled damages, 6 classes). Request access at https://cardd-ustc.github.io and upload `CarDD_release.zip` to your Google Drive. Any COCO-format damage dataset (Roboflow, CVAT export) works too.

In [ ]:
!nvidia-smi
!git clone https://github.com/MIsmail2524/DentBid.git
%cd DentBid/ml
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/CarDD_release.zip -d /content/data

In [ ]:
!python scripts/prepare_dataset.py --coco-root /content/data/CarDD_release/CarDD_COCO --out datasets/damage

## 1. Damage model
`yolo11m-seg` at 1024 px takes ~3-4 h on a T4. Use `yolo11s-seg.pt` for a faster, smaller model or `yolo11l-seg.pt` for higher accuracy.

In [ ]:
!python scripts/train.py damage --data datasets/damage/data.yaml --model yolo11m-seg.pt

In [ ]:
!python scripts/evaluate.py --weights weights/damage.pt --data datasets/damage/data.yaml --split test

## 2. Car-parts model
Uses the Ultralytics `carparts-seg` dataset (23 parts), downloaded automatically. Parts let DentBid say *where* the damage is and convert pixels to centimetres.

In [ ]:
!python scripts/train.py parts --model yolo11s-seg.pt

## 3. Try it and save weights

In [ ]:
!python scripts/predict.py datasets/damage/images/test/*.jpg --damage weights/damage.pt --parts weights/parts.pt 2>/dev/null | tail -30

In [ ]:
!mkdir -p /content/drive/MyDrive/dentbid_weights && cp weights/*.pt weights/*.json /content/drive/MyDrive/dentbid_weights/ 2>/dev/null; ls -la /content/drive/MyDrive/dentbid_weights

Copy `damage.pt` and `parts.pt` into `ml/weights/` on the server, then restart the backend.